# Week 10 Lab — Meeting MODFLOW

**HWRS 564a · Fall 2026**

For nine weeks you have been inferring what an aquifer is doing from patterns in
observations. Now you build a model of it.

MODFLOW is a Fortran program that solves the groundwater flow equation on a grid.
FloPy is the Python library that writes its input files, runs it, and reads its
output back. **You will never edit a MODFLOW input file by hand in this course** —
but you will read a few, because knowing what FloPy is writing is what lets you
debug it.

This week is orientation: what the pieces are, how to run one, and how to tell
whether the answer means anything.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without complaint, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.


## Learning objectives

By the end of this notebook you can:

1. Say what each of the six core MODFLOW packages answers
2. Confirm the MODFLOW binary exists before trying to use it
3. Build, write, and run the simplest possible model from FloPy
4. Check `success` properly, and get a useful message when it fails
5. Read heads out of the binary `.hds` file
6. Find the water budget in the `.list` file and check that it closes

---

## Part 1 — Is MODFLOW even here?

MODFLOW is a compiled binary, not a Python package. `postbuild.sh` downloaded it
when your codespace was created. **Check for it before anything else** — a
missing binary produces a failure much later and much less clearly.

In [ ]:
from pathlib import Path

import flopy
import matplotlib.pyplot as plt
import numpy as np

ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
MF_EXE = ROOT / "modflow" / "mf2005"

assert MF_EXE.exists(), (
    f"MODFLOW binary not found at {MF_EXE}. Your codespace may not have "
    "finished building — run ./postbuild.sh from a terminal."
)

print(f"flopy {flopy.__version__}")
print(f"MODFLOW at {MF_EXE.relative_to(ROOT)}")
print(f"other binaries available: {sorted(p.name for p in MF_EXE.parent.iterdir())[:8]}")

That assertion, with that message, goes at the top of every MODFLOW notebook you
write. It costs one line and it turns "`FileNotFoundError` from somewhere inside
flopy" into "run postbuild.sh".

### Where the model files go

A MODFLOW run writes about a dozen files. **Give every model its own scratch
directory**, outside version control — otherwise `.hds`, `.cbc` and `.list`
files accumulate in the repo root and get committed.

In [ ]:
WS = ROOT / "_run" / "week10_first"
WS.mkdir(parents=True, exist_ok=True)

print(f"model workspace: {WS.relative_to(ROOT)}")
print("(_run/ is in .gitignore — model output is never committed)")

---

## Part 2 — Six questions, six packages

MODFLOW has no single "model file". It has a **package per question**, and a
`.nam` file listing them.

| Question | Package | FloPy class |
|---|---|---|
| What shape is the domain, in what units, over what time? | `DIS` | `ModflowDis` |
| Which cells are active, and where do we start? | `BAS` | `ModflowBas` |
| How permeable is it? | `LPF` | `ModflowLpf` |
| How do we solve it? | `PCG` | `ModflowPcg` |
| What do we save? | `OC` | `ModflowOc` |
| Who is pumping, recharging, evaporating? | `WEL`, `RCH`, `EVT`, … | one class each |

That table is the whole mental model. Everything you build from here is those
answers, plus whichever stress packages the problem needs.

### The simplest useful model

One layer, 10 by 20 cells, constant head on the east and west edges, nothing
else. Flow runs west to east because we told the two edges to differ.

In [ ]:
NLAY, NROW, NCOL = 1, 10, 20
DELR = DELC = 100.0        # m, cell size

mf = flopy.modflow.Modflow("first", model_ws=str(WS), exe_name=str(MF_EXE))

# DIS — geometry and time
flopy.modflow.ModflowDis(
    mf, NLAY, NROW, NCOL,
    delr=DELR, delc=DELC,
    top=100.0, botm=0.0,       # ELEVATIONS, not thicknesses
    nper=1, steady=True,
)
print(mf.dis)

`top=100.0, botm=0.0` are **elevations above datum**, so this aquifer is 100 m
thick. Passing thicknesses instead gives you a layer with a negative thickness
and an error message that does not mention `botm`.

In [ ]:
# BAS — which cells are active, and the starting heads
ibound = np.ones((NLAY, NROW, NCOL), dtype=int)
ibound[:, :, 0] = -1        # west edge: constant head
ibound[:, :, -1] = -1       # east edge: constant head

strt = np.full((NLAY, NROW, NCOL), 95.0)
strt[:, :, -1] = 85.0       # 10 m lower on the east side

flopy.modflow.ModflowBas(mf, ibound=ibound, strt=strt)

print("ibound values used:", np.unique(ibound))
print("  1 = active,  0 = inactive,  -1 = constant head")

> **`strt` does two jobs.** In active cells it is only a starting guess for the
> solver. In `ibound == -1` cells it *is* the boundary condition, and it does not
> change. Same array, two meanings, decided by `ibound`.

In [ ]:
# LPF — hydraulic properties.  PCG — the solver.  OC — what to save.
flopy.modflow.ModflowLpf(mf, hk=10.0, laytyp=0, ipakcb=53)
flopy.modflow.ModflowPcg(mf)
flopy.modflow.ModflowOc(
    mf, stress_period_data={(0, 0): ["save head", "save budget", "print budget"]}
)

print("packages attached:", mf.get_package_list())

Two arguments worth naming now, because they come back every week:

- **`laytyp=0`** means *confined* — transmissivity is fixed. `laytyp=1` means
  *convertible*, so transmissivity depends on the computed head and the problem
  becomes nonlinear. Week 12 is about that difference.
- **`ipakcb=53`** tells the package to write its flows to the cell-budget file.
  Leave it out and the budget terms for that package are silently absent.

### YOUR TURN 1

Before running anything, work out what this model *should* say.

The two constant-head boundaries are 95 m and 85 m, and there are 20 columns of
100 m cells. With no recharge and no wells, the head profile must be a straight
line between them.

Make a **first estimate** using the full width of the domain:

- `domain_length_m` — the east–west extent of the model
- `expected_gradient` — the hydraulic gradient, dimensionless
- `expected_q` — the specific discharge in m/d, from Darcy's law with `hk=10.0`

Hold onto these. Part 5 checks them against MODFLOW and the answer is *not*
quite what you get here — for a reason worth knowing.

In [ ]:
hk = 10.0

# YOUR TURN
domain_length_m = ...
expected_gradient = ...
expected_q = ...

In [ ]:
# CHECK
assert abs(domain_length_m - 2000.0) < 1e-9, f"got {domain_length_m}"
assert abs(expected_gradient - 0.005) < 1e-9, f"got {expected_gradient}"
assert abs(expected_q - 0.05) < 1e-9, f"got {expected_q}"
print(f"domain      {domain_length_m:.0f} m")
print(f"gradient    {expected_gradient:.4f}")
print(f"q           {expected_q:.3f} m/d")
print("\nThat is the arithmetic. Part 5 asks whether MODFLOW agrees.")

---

## Part 3 — Running it, and checking that it ran

`write_input()` turns the FloPy objects into text files. `run_model()` shells out
to the binary.

In [ ]:
mf.write_input()

for f in sorted(WS.iterdir()):
    print(f"  {f.name:16s} {f.stat().st_size:8,d} bytes")

In [ ]:
success, buff = mf.run_model(silent=True, report=True)

assert success, "MODFLOW did not converge:\n" + "\n".join(buff[-20:])
print(f"converged. {len(buff)} lines of output captured.")

> **Three details in those two lines, and all three matter.**
>
> **FloPy does not raise when a model fails.** It returns `(success, buff)` and
> carries on. Ignore `success` and you will happily plot whatever was last
> written to the `.hds` file and present it as a result.
>
> **`report=True` is not optional.** With `silent=True` alone, FloPy returns an
> **empty** `buff` — so the assertion fires with nothing in the message, at
> exactly the moment you needed a diagnostic. You will see tutorials that omit
> it.
>
> **`silent=True` only suppresses the console echo.** The full run log is always
> written to the `.list` file, which is where the real diagnosis lives.

### YOUR TURN 2 — break it on purpose

You need to have seen a failure before you meet one by accident.

Build the same model again, but hand the solver an impossible convergence
criterion: `mxiter=1, iter1=1, hclose=1e-12`. One iteration cannot close a head
change to a picometre.

Capture `success` and `buff` **without** asserting, so the cell completes.

In [ ]:
WS_FAIL = ROOT / "_run" / "week10_fail"
WS_FAIL.mkdir(parents=True, exist_ok=True)

bad = flopy.modflow.Modflow("nope", model_ws=str(WS_FAIL), exe_name=str(MF_EXE))
flopy.modflow.ModflowDis(bad, NLAY, NROW, NCOL, delr=DELR, delc=DELC,
                         top=100.0, botm=0.0, nper=1, steady=True)
flopy.modflow.ModflowBas(bad, ibound=ibound, strt=strt)
flopy.modflow.ModflowLpf(bad, hk=hk, laytyp=1, ipakcb=53)
flopy.modflow.ModflowPcg(bad, mxiter=1, iter1=1, hclose=1e-12, rclose=1e-12)
flopy.modflow.ModflowOc(bad, stress_period_data={(0, 0): ["save head"]})
bad.write_input()

# YOUR TURN
bad_success, bad_buff = ...

In [ ]:
# CHECK
assert bad_success is False, f"this model should NOT have converged (got {bad_success})"
assert len(bad_buff) > 0, (
    "buff is empty — you need report=True, or the failure message tells you nothing"
)
print(f"success = {bad_success}, with {len(bad_buff)} lines of output")
print("\nlast few lines:")
for line in [l.strip() for l in bad_buff if l.strip()][-3:]:
    print(f"  {line}")

There it is: **FAILED TO MEET SOLVER CONVERGENCE CRITERIA**.

The `.list` file says more, including the number that tells you *how* badly it
failed:

In [ ]:
listing = (WS_FAIL / "nope.list").read_text().splitlines()
tail = [l.strip() for l in listing if l.strip()]

for line in tail[-6:]:
    print(line)

**BUDGET PERCENT DISCREPANCY IS -162.7.** Water is not being conserved, by a
factor of more than two. That single number is the first thing to look at when a
model misbehaves, and we come back to it in Part 5.

---

## Part 4 — Reading the heads back

Heads are written to a binary file. `flopy.utils.HeadFile` reads it.

In [ ]:
hds = flopy.utils.HeadFile(str(WS / "first.hds"))

print(f"times available: {hds.get_times()}")
head = hds.get_data()                     # last time step by default
print(f"shape: {head.shape}   (layers, rows, columns)")
print(f"range: {head.min():.2f} to {head.max():.2f} m")

### YOUR TURN 3

Confirm MODFLOW agrees with the arithmetic you did in Part 1.

Extract the head profile along **row 5** and compare it with a straight line
between its two ends.

- `profile` — the heads along row 5 of layer 0, as a 1D array
- `max_deviation` — the largest absolute difference between `profile` and a
  straight line joining `profile[0]` to `profile[-1]`

`np.linspace(a, b, n)` builds that straight line for you.

In [ ]:
# YOUR TURN
profile = ...
max_deviation = ...

In [ ]:
# CHECK
assert profile.shape == (20,), f"expected 20 values, got {profile.shape}"
assert abs(profile[0] - 95.0) < 1e-6 and abs(profile[-1] - 85.0) < 1e-6, \
    "the ends should sit exactly on the constant heads"
assert max_deviation < 1e-4, f"expected a straight line, deviation was {max_deviation}"
print(f"profile ends: {profile[0]:.2f} -> {profile[-1]:.2f} m")
print(f"max deviation from a straight line: {max_deviation:.2e} m")
print("\nMODFLOW reproduced the analytic solution exactly. Correct.")

**This is the most important habit in the whole module.** Before you trust a
model on a problem you *can't* solve by hand, run it on one you can.

A one-dimensional confined aquifer between two fixed heads has a linear
solution, and MODFLOW reproduces it to within floating-point noise. Now the
grid, the units, the boundary handling and the solver are all confirmed working
— so when a later result surprises you, the surprise is about the *physics* you
added, not about whether you set the model up correctly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

mv = flopy.plot.PlotMapView(model=mf, ax=axes[0])
band = mv.plot_array(head[0], cmap="Blues_r")
cs = mv.contour_array(head[0], levels=np.arange(86, 96, 1.0), colors="k", linewidths=0.7)
axes[0].clabel(cs, fmt="%.0f", fontsize=8)
axes[0].set_xlabel("easting (m)")
axes[0].set_ylabel("northing (m)")
axes[0].set_title("Head, plan view")
fig.colorbar(band, ax=axes[0], label="head (m)", shrink=0.85)

x = (np.arange(NCOL) + 0.5) * DELR
axes[1].plot(x, profile, "o-", color="#AB0520", ms=4, label="MODFLOW")
axes[1].plot(x, np.linspace(profile[0], profile[-1], NCOL), "--",
             color="#0C234B", lw=1.5, label="analytic (linear)")
axes[1].set_xlabel("distance east (m)")
axes[1].set_ylabel("head (m)")
axes[1].set_title("Row 5 profile vs. the analytic solution")
axes[1].legend(frameon=False)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## Part 5 — The water budget

Every MODFLOW run writes a mass balance to the `.list` file. **Read it every
time.** A model can converge and still be wrong; a budget that doesn't close
tells you so.

In [ ]:
listing = (WS / "first.list").read_text().splitlines()

start = next(i for i, l in enumerate(listing) if "VOLUMETRIC BUDGET" in l)
for line in listing[start:start + 26]:
    if line.strip():
        print(line.rstrip())

Two halves — `IN` and `OUT` — and at the bottom the number that matters:
**PERCENT DISCREPANCY**.

In [ ]:
discrepancy = next(
    float(l.split()[-1]) for l in reversed(listing) if "PERCENT DISCREPANCY" in l
)
print(f"percent discrepancy: {discrepancy:.4f} %")

assert abs(discrepancy) < 0.1, "a converged steady-state model should close to well under 0.1%"
print("The budget closes. Water in equals water out.")

### YOUR TURN 4

Write `budget_discrepancy(list_path)`, returning the percent discrepancy from a
MODFLOW `.list` file as a float.

Read the file, find the **last** line containing `"PERCENT DISCREPANCY"`, and
pull the number off the end of it. You will use this function every week from
here.

One wrinkle, and it is the kind real file formats always have. MODFLOW writes
that phrase two different ways:

```
 PERCENT DISCREPANCY =        -162.74     PERCENT DISCREPANCY =        -162.74
 BUDGET PERCENT DISCREPANCY IS -162.7401
```

One uses `=` and repeats itself in two columns; the other uses `IS`. Splitting
on `"="` works on the first and crashes on the second. **Take the last
whitespace-separated token instead** — that is correct for both.

In [ ]:
# YOUR TURN
def budget_discrepancy(list_path):
    """Percent mass-balance discrepancy from a MODFLOW .list file."""
    ...

In [ ]:
# CHECK
good = budget_discrepancy(WS / "first.list")
assert isinstance(good, float), f"should return a float, got {type(good)}"
assert abs(good) < 0.1, f"the converged model should close: got {good}"

bad_value = budget_discrepancy(WS_FAIL / "nope.list")
assert abs(bad_value) > 100, f"the failed model was off by 160%, got {bad_value}"

print(f"converged model : {good:+.4f} %")
print(f"failed model    : {bad_value:+.4f} %")
print("Correct.")

### YOUR TURN 5 — how much water is actually moving?

MODFLOW's budget reports the flow through the system as the `CONSTANT HEAD`
term. Compare it with the Darcy estimate from Part 1.

- `cross_section_area` — the full north–south width times the aquifer thickness
- `darcy_flow` — `expected_q` times that area

In [ ]:
thickness_m = 100.0

# YOUR TURN
cross_section_area = ...
darcy_flow = ...

In [ ]:
# CHECK
assert abs(cross_section_area - 100_000.0) < 1e-6, f"got {cross_section_area}"
assert abs(darcy_flow - 5000.0) < 1e-6, f"got {darcy_flow}"

chd = next(
    float(l.split()[-1]) for l in listing if l.strip().startswith("CONSTANT HEAD =")
)
print(f"Darcy's law (Part 1) says {darcy_flow:9.1f} m3/d")
print(f"MODFLOW's budget says     {chd:9.1f} m3/d")
print(f"difference                {100 * (chd - darcy_flow) / darcy_flow:+9.1f} %")
print("\nCorrect — and that 5% gap is not an error. Read on.")

### Where the 5% went

MODFLOW is right and the Part 1 arithmetic is wrong, in a way that catches
everybody once.

**A constant head is specified at the cell *centre*, not at the model edge.** The
head of 95 m sits at the middle of column 0, and the 85 m sits at the middle of
column 19. The distance the water actually travels between those two points is

$$19 	imes 100\ 	ext{m} = 1900\ 	ext{m}$$

not the 2000 m width of the domain. There is half a cell of aquifer outside each
boundary head that is not part of the flow path.

### YOUR TURN 6

Redo it with the right distance.

- `centre_to_centre_m` — the distance between the two constant-head cell centres
- `corrected_q` — specific discharge over that distance
- `corrected_flow` — and the volumetric flow

In [ ]:
# YOUR TURN
centre_to_centre_m = ...
corrected_q = ...
corrected_flow = ...

In [ ]:
# CHECK
assert abs(centre_to_centre_m - 1900.0) < 1e-9, f"got {centre_to_centre_m}"
assert abs(corrected_flow - chd) < 0.1, (
    f"expected to match MODFLOW's {chd:.2f}, got {corrected_flow:.2f}"
)
print(f"centre-to-centre distance {centre_to_centre_m:9.0f} m")
print(f"corrected q               {corrected_q:9.5f} m/d")
print(f"corrected flow            {corrected_flow:9.2f} m3/d")
print(f"MODFLOW                   {chd:9.2f} m3/d")
print("\nExact agreement. Correct.")

**This is why you test against an analytic solution.** A 5% error is small enough
to look like rounding and large enough to matter, and the only way you would ever
have found it is by predicting the answer first and being bothered when the
numbers didn't match.

The practical consequence: **a boundary condition is half a cell inside the edge
you drew.** On a 250 m grid that is 125 m of slop, and if you are trying to match
an observed gradient across a small domain it is the first thing to check.

**Think about this before Thursday:** you now have a model that reproduces an
analytic solution, closes its budget, and agrees with Darcy's law. That is the
baseline against which everything else this module gets judged.

Next week you put a well in it, and the head field stops being something you
could have worked out on paper.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

- **HW 8 — Regression analysis**, Wednesday 10/28 at 11:59pm
- **Project 2 analysis, part 2**

## Next week

Building a real 2D model: a well, recharge, and a head field you cannot predict
by hand.

## Stuck?

- `FileNotFoundError` mentioning `mf2005` — the binary isn't there. Run
  `./postbuild.sh` from a terminal.
- `success` is `False` and `buff` is empty — you left out `report=True`.
- A model that converges but whose heads are all identical to `strt` usually
  means every cell is `ibound == -1`, so nothing was free to change.
- `AssertionError` on layer thickness, or a `.list` file complaining about
  `BOTM`, means `top` and `botm` are the wrong way round.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.